# TCC — Patch Sentinel-2 de 512 × 512 em Monte Belo

Ative Internet e GPU T4. Anexe o backup privado completo da U-Net e da U-Mamba. O recorte é selecionado dentro de Monte Belo, município da RGI fora da base Emater usada no treino. A execução usa os mesmos pesos treinados com 256 px e faz uma inferência de 512 px por rede. A imagem RGB recebe ajuste de contraste para a figura. O resultado é visual, sem máscara de referência.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

CODE_REF = "fce8437a03bacf9ee5f1712d8812951b0a53ff81"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "tcc_nova_cena"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/oguel/tcc-umamba.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Preserve as alterações no checkout antes de continuar.")
subprocess.run(["git", "fetch", "origin", CODE_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_REF
print("Código fixado:", CODE_REF)


## Localizar os backups corretos

A célula verifica os hashes dos melhores pesos. Se o backup estiver compactado, aceita somente o ZIP completo já auditado e extrai para uma pasta nova de trabalho. Não modifica as entradas ou os checkpoints originais.

In [ ]:
EXPECTED = {
    "unet": "addd3b5f680c1495dfa73b7eae6332fe085015c8d3e7035d825dfab4adc1d2cf",
    "umamba": "038a100adc397ae7482bf12550f958c9046829d5ff3d4f16d56275f69a91f9bf",
}
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

roots = [WORK / "tcc_runs", INPUT, WORK / "backup_final_extraido"]
def locate():
    runs = {}
    for kind, expected in EXPECTED.items():
        candidates = [p for root in roots if root.exists() for p in root.rglob(f"best_{kind}_rgi.pt")]
        for path in sorted(candidates):
            if sha256(path) == expected:
                runs[kind] = path.parent
                break
    return runs

RUNS = locate()
if len(RUNS) != 2:
    candidates = [p for p in INPUT.rglob("*.zip") if "backup" in p.name.lower() and p.stat().st_size == 53710616]
    complete = [p for p in candidates if sha256(p) == "234cb97b21931b4c3b9ec8140003b17ca5319a03cfa899b297140160a6b61a1a"]
    if complete:
        destination = WORK / "backup_final_extraido"
        if destination.exists():
            raise RuntimeError("Extração anterior incompleta: preserve a pasta e revise o problema.")
        with zipfile.ZipFile(complete[0]) as archive:
            entries = archive.infolist()
            if len(entries) > 500 or sum(e.file_size for e in entries) > 512 * 1024**2:
                raise RuntimeError("Dimensão do arquivo inesperada.")
            for entry in entries:
                resolved = (destination / entry.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise RuntimeError("Caminho inválido no ZIP.")
            if archive.testzip() is not None:
                raise RuntimeError("ZIP corrompido.")
            destination.mkdir()
            archive.extractall(destination)
        RUNS = locate()
if len(RUNS) != 2:
    raise RuntimeError("Anexe como entrada privada o backup_COMPLETO_UNET_UMAMBA.zip ou as duas pastas de execução completas.")
print({kind: str(path) for kind, path in RUNS.items()})


## Preparar o ambiente CUDA

A instalação e a inferência ocorrem em processos separados.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-runtime.txt"), "geopandas"], check=True)
subprocess.run([sys.executable, "-c", "from src.models.umamba_runtime import install_prebuilt_colab_stack; print('PyTorch alterado:', install_prebuilt_colab_stack())"], cwd=REPO, check=True)


## Gerar o patch e comparar as previsões

In [ ]:
script = WORK / "infer_patch512.py"
script.write_text('import argparse\nimport json\nfrom pathlib import Path\n\nimport geopandas as gpd\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport rasterio\nimport torch\nfrom rasterio.transform import from_origin\nfrom shapely.geometry import box\n\nfrom scripts.infer_new_sentinel import (\n    BANDS, CRS, IBGE, CHECKPOINTS, THRESHOLD, UNet, _file_sha256,\n    fetch_json, grid_mask, load_mosaic, region_geometry, write_tiff,\n)\nfrom src.models.umamba import build_official_umamba_enc_2d\nfrom src.models.umamba_runtime import ensure_umamba_runtime\n\nSIZE = 512\nPIXEL = 10\nMUNICIPALITY = "Monte Belo"\nMUNICIPALITY_ID = 3143005\nBOUNDARY_URL = (\n    f"{IBGE}/v4/malhas/municipios/{MUNICIPALITY_ID}"\n    "?formato=application%2Fvnd.geo%2Bjson"\n)\n\n\ndef select_grid(area):\n    west, south, east, north = area.bounds\n    center = area.centroid\n    span = SIZE * PIXEL\n    candidates = []\n    for left in range(int(np.ceil(west / 640)) * 640, int(east - span) + 1, 640):\n        for bottom in range(int(np.ceil(south / 640)) * 640, int(north - span) + 1, 640):\n            window = box(left, bottom, left + span, bottom + span)\n            if area.covers(window):\n                candidates.append((center.distance(window.centroid), left, bottom))\n    if not candidates:\n        raise RuntimeError("Nenhuma janela de 512 px cabe integralmente no município.")\n    _, left, bottom = min(candidates)\n    return {\n        "crs": CRS, "width": SIZE, "height": SIZE,\n        "transform": from_origin(left, bottom + span, PIXEL, PIXEL),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("unet", type=Path)\n    parser.add_argument("umamba", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    boundary = fetch_json(BOUNDARY_URL, timeout=60)\n    municipality = gpd.GeoDataFrame.from_features(boundary, crs="EPSG:4326").to_crs(CRS)\n    area = municipality.geometry.union_all().intersection(region_geometry())\n    grid = select_grid(area)\n    inside = grid_mask(area, grid)\n    image, source, scenes, _ = load_mosaic(grid, area, inside, None, [])\n    valid = source > 0\n    if valid.mean() < 0.99:\n        raise RuntimeError(f"Cobertura válida abaixo de 99%: {valid.mean():.2%}.")\n    runtime = ensure_umamba_runtime()\n\n    probabilities = {}\n    for name in ("unet", "umamba"):\n        path = getattr(args, name) / f"best_{name}_rgi.pt"\n        if _file_sha256(path) != CHECKPOINTS[name]:\n            raise ValueError(f"Checkpoint inesperado: {name}")\n        model = (\n            UNet(in_channels=4, out_channels=1)\n            if name == "unet"\n            else build_official_umamba_enc_2d(input_channels=4, num_classes=1, input_size=(512, 512))\n        )\n        checkpoint = torch.load(path, weights_only=True, map_location="cpu")\n        model.load_state_dict(checkpoint["model_state_dict"], strict=True)\n        model = model.cuda().eval()\n        with torch.inference_mode():\n            logits = model(torch.from_numpy(image[None]).cuda())\n            if logits.shape != (1, 1, SIZE, SIZE) or not torch.isfinite(logits).all():\n                raise ValueError(f"Saída inválida: {name}")\n            probabilities[name] = logits.sigmoid()[0, 0].cpu().numpy()\n        del model, checkpoint, logits\n\n    args.output.mkdir()\n    with rasterio.open(\n        args.output / "patch_512.tif", "w", driver="GTiff", count=4,\n        dtype="float32", nodata=-9999, compress="deflate", **grid,\n    ) as target:\n        target.write(np.where(valid[None], image, -9999))\n        target.descriptions = tuple(BANDS.values())\n    write_tiff(args.output / "cena_por_pixel.tif", source, grid, 0)\n\n    rgb = image[[2, 1, 0]].transpose(1, 2, 0)\n    low, high = np.percentile(rgb[valid], [2, 98])\n    rgb = np.clip((rgb - low) / max(high - low, 1e-6), 0, 1)\n    rgb[~valid] = 0.33\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4))\n    axes[0].imshow(rgb, interpolation="nearest")\n    axes[0].set_title("Sentinel-2 | Monte Belo")\n    for ax, (name, probability) in zip(axes[1:], probabilities.items(), strict=True):\n        predicted = probability >= THRESHOLD\n        write_tiff(\n            args.output / f"{name}_probabilidade.tif", np.where(valid, probability, -9999),\n            grid, -9999,\n        )\n        write_tiff(\n            args.output / f"{name}_cafe.tif", np.where(valid, predicted, 255).astype("uint8"),\n            grid, 255,\n        )\n        ax.imshow(rgb, interpolation="nearest")\n        overlay = np.zeros((SIZE, SIZE, 4))\n        overlay[valid & predicted] = [0.1, 0.85, 0.25, 0.55]\n        ax.imshow(overlay, interpolation="nearest")\n        ax.set_title("U-Net" if name == "unet" else "U-Mamba")\n    for ax in axes:\n        ax.axis("off")\n    fig.tight_layout()\n    fig.savefig(args.output / "comparacao.png", dpi=150)\n    plt.close(fig)\n    report = {\n        "stage": "unlabeled_512_inference",\n        "training_patch_size": 256, "inference_patch_size": SIZE,\n        "municipality": MUNICIPALITY, "municipality_ibge": MUNICIPALITY_ID,\n        "boundary_url": BOUNDARY_URL,\n        "crs": CRS, "bounds": rasterio.transform.array_bounds(SIZE, SIZE, grid["transform"]),\n        "valid_fraction": float(valid.mean()), "threshold": THRESHOLD,\n        "scenes": [{**entry, "stac": entry["stac"]["id"]} for entry in scenes],\n        "checkpoints": CHECKPOINTS, "runtime": runtime,\n    }\n    (args.output / "relatorio.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = WORK / f"monte_belo_patch512_{stamp}"
import os
env = {**os.environ, "PYTHONPATH": str(REPO)}
subprocess.run([
    sys.executable, "-u", str(script), str(RUNS["unet"]),
    str(RUNS["umamba"]), str(output),
], cwd=REPO, env=env, check=True)
archive = shutil.make_archive(str(output), "zip", root_dir=output)
print("Baixar:", archive)
from IPython.display import Image, display
display(Image(filename=str(output / "comparacao.png")))
